# Income Time Series Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

## Data visualization

In [ ]:
df = pd.read_csv('income_data.csv')

df['Date'] = pd.to_datetime(df['Date'])

df = df.sort_values(by='Date')

df.set_index('Date', inplace=True)

plt.figure(figsize=(12, 6))

plt.plot(df.index, df['Income'], color='#1f77b4', linewidth=1.5, label='Daily Income')

plt.title('Daily Store Income', fontsize=16, pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Income', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.6)

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

## Information extraction

In [ ]:
window_size = 7  # Adjust as needed (e.g., 7 for weekly, 30 for monthly)
df['Rolling_Mean'] = df['Income'].rolling(window=window_size).mean()
df['Rolling_Var'] = df['Income'].rolling(window=window_size).var()

# 2. Create a 2-panel plot (Top: Income + Mean, Bottom: Variance)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# --- Top Plot: Daily Income + Rolling Mean ---
ax1.plot(df.index, df['Income'], color='#1f77b4', alpha=0.4, linewidth=1, label='Daily Income')
ax1.plot(df.index, df['Rolling_Mean'], color='#d62728', linewidth=2, label=f'{window_size}-Day Rolling Mean')
ax1.set_title('Daily Income & Rolling Mean', fontsize=14)
ax1.set_ylabel('Income', fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper left')

# --- Bottom Plot: Rolling Variance ---
ax2.plot(df.index, df['Rolling_Var'], color='#2ca02c', linewidth=1.5, label=f'{window_size}-Day Rolling Variance')
ax2.set_title(f'{window_size}-Day Rolling Variance (Volatility)', fontsize=14)
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Variance', fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='upper left')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Monthly mean vs variance

In [ ]:
# Group data by Month
monthly_stats = df['Income'].resample('ME').agg(['mean', 'var'])

# Plot Variance vs. Mean
plt.figure(figsize=(8, 5))
plt.scatter(monthly_stats['mean'], monthly_stats['var'], color='#8c564b', alpha=0.8, edgecolors='k')

plt.title('Monthly Mean-Variance Scatter Plot', fontsize=14)
plt.xlabel('Monthly Mean Income', fontsize=12)
plt.ylabel('Monthly Income Variance', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

Box-Cow transformation parameter

In [ ]:
from scipy import stats

# Add a shift of +1 to handle zeros (0 becomes 1, so log(1) = 0)
income_shifted = df['Income'] + 1

# Calculate Box-Cox lambda on shifted data
transformed_data, fitted_lambda = stats.boxcox(income_shifted)

print(f"Optimal Lambda (Shifted +1): {fitted_lambda:.3f}")

Lambda is close to 1, so no log transformation is needed.

## Seasonality

### Monthly

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sm.graphics.tsa.month_plot(df['Income'].resample('ME').mean(), ax=ax)

ax.set_title('Month Plot (Monthly Mean Income)', fontsize=14)
ax.set_ylabel('Mean Daily Income', fontsize=12)
plt.show()


df['Month'] = df.index.month_name()
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']

plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='Month', y='Income', order=month_order, palette='Blues')
plt.title('Monthly Income Distribution (Seasonality Check)')
plt.xticks(rotation=45)
plt.show()